In [ ]:
import numpy as np

In [ ]:
class LSTMRegressor:
    def __init__(self, input_size, hidden_size=16, output_size=1, learning_rate=0.01, epochs=300, random_state=42):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.random_state = random_state
        self.loss_history = []
        self._initialize_parameters()

    def _initialize_parameters(self):
        np.random.seed(self.random_state)
        z_size = self.hidden_size + self.input_size

        self.Wf = np.random.randn(self.hidden_size, z_size) * 0.1
        self.Wi = np.random.randn(self.hidden_size, z_size) * 0.1
        self.Wc = np.random.randn(self.hidden_size, z_size) * 0.1
        self.Wo = np.random.randn(self.hidden_size, z_size) * 0.1

        self.bf = np.zeros((self.hidden_size, 1))
        self.bi = np.zeros((self.hidden_size, 1))
        self.bc = np.zeros((self.hidden_size, 1))
        self.bo = np.zeros((self.hidden_size, 1))

        self.Wy = np.random.randn(self.output_size, self.hidden_size) * 0.1
        self.by = np.zeros((self.output_size, 1))

    def fit(self, X, y):
        X = np.array(X, dtype=float)
        y = np.array(y, dtype=float).reshape(-1, self.output_size)

        for _ in range(self.epochs):
            total_loss = 0

            for i in range(len(X)):
                caches, y_pred = self._forward(X[i])
                loss = np.mean((y_pred - y[i].reshape(-1, 1)) ** 2)
                total_loss += loss
                grads = self._backward(X[i], y[i].reshape(-1, 1), caches, y_pred)
                self._update_parameters(grads)

            self.loss_history.append(total_loss / len(X))

    def predict(self, X):
        X = np.array(X, dtype=float)
        preds = []

        for i in range(len(X)):
            _, y_pred = self._forward(X[i])
            preds.append(y_pred.ravel())

        return np.array(preds)

    def mse(self, X, y):
        y = np.array(y, dtype=float).reshape(-1, self.output_size)
        y_pred = self.predict(X)
        return np.mean((y_pred - y) ** 2)

    def rmse(self, X, y):
        return np.sqrt(self.mse(X, y))

    def _forward(self, x_seq):
        h_prev = np.zeros((self.hidden_size, 1))
        c_prev = np.zeros((self.hidden_size, 1))
        caches = []

        for t in range(len(x_seq)):
            x_t = x_seq[t].reshape(-1, 1)
            z = np.vstack((h_prev, x_t))

            f_t = self._sigmoid(self.Wf @ z + self.bf)
            i_t = self._sigmoid(self.Wi @ z + self.bi)
            c_hat_t = np.tanh(self.Wc @ z + self.bc)
            c_t = f_t * c_prev + i_t * c_hat_t
            o_t = self._sigmoid(self.Wo @ z + self.bo)
            h_t = o_t * np.tanh(c_t)

            cache = {
                "z": z,
                "f": f_t,
                "i": i_t,
                "c_hat": c_hat_t,
                "c": c_t,
                "o": o_t,
                "h": h_t,
                "h_prev": h_prev,
                "c_prev": c_prev
            }

            caches.append(cache)
            h_prev = h_t
            c_prev = c_t

        y_pred = self.Wy @ h_prev + self.by
        return caches, y_pred

    def _backward(self, x_seq, y_true, caches, y_pred):
        dWf = np.zeros_like(self.Wf)
        dWi = np.zeros_like(self.Wi)
        dWc = np.zeros_like(self.Wc)
        dWo = np.zeros_like(self.Wo)

        dbf = np.zeros_like(self.bf)
        dbi = np.zeros_like(self.bi)
        dbc = np.zeros_like(self.bc)
        dbo = np.zeros_like(self.bo)

        dWy = np.zeros_like(self.Wy)
        dby = np.zeros_like(self.by)

        dy = 2 * (y_pred - y_true) / y_true.shape[0]

        dWy += dy @ caches[-1]["h"].T
        dby += dy

        dh_next = self.Wy.T @ dy
        dc_next = np.zeros((self.hidden_size, 1))

        for t in reversed(range(len(caches))):
            cache = caches[t]

            z = cache["z"]
            f_t = cache["f"]
            i_t = cache["i"]
            c_hat_t = cache["c_hat"]
            c_t = cache["c"]
            o_t = cache["o"]
            c_prev = cache["c_prev"]

            tanh_c = np.tanh(c_t)

            dh = dh_next
            do = dh * tanh_c
            do_raw = do * o_t * (1 - o_t)

            dc = dh * o_t * (1 - tanh_c ** 2) + dc_next
            df = dc * c_prev
            df_raw = df * f_t * (1 - f_t)

            di = dc * c_hat_t
            di_raw = di * i_t * (1 - i_t)

            dc_hat = dc * i_t
            dc_hat_raw = dc_hat * (1 - c_hat_t ** 2)

            dWf += df_raw @ z.T
            dWi += di_raw @ z.T
            dWc += dc_hat_raw @ z.T
            dWo += do_raw @ z.T

            dbf += df_raw
            dbi += di_raw
            dbc += dc_hat_raw
            dbo += do_raw

            dz = (
                self.Wf.T @ df_raw +
                self.Wi.T @ di_raw +
                self.Wc.T @ dc_hat_raw +
                self.Wo.T @ do_raw
            )

            dh_next = dz[:self.hidden_size, :]
            dc_next = dc * f_t

        grads = {
            "Wf": dWf,
            "Wi": dWi,
            "Wc": dWc,
            "Wo": dWo,
            "bf": dbf,
            "bi": dbi,
            "bc": dbc,
            "bo": dbo,
            "Wy": dWy,
            "by": dby
        }

        return grads

    def _update_parameters(self, grads):
        self.Wf -= self.learning_rate * grads["Wf"]
        self.Wi -= self.learning_rate * grads["Wi"]
        self.Wc -= self.learning_rate * grads["Wc"]
        self.Wo -= self.learning_rate * grads["Wo"]

        self.bf -= self.learning_rate * grads["bf"]
        self.bi -= self.learning_rate * grads["bi"]
        self.bc -= self.learning_rate * grads["bc"]
        self.bo -= self.learning_rate * grads["bo"]

        self.Wy -= self.learning_rate * grads["Wy"]
        self.by -= self.learning_rate * grads["by"]

    def _sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

In [ ]:
X = np.array([
    [[0.1], [0.2], [0.3]],
    [[0.2], [0.1], [0.4]],
    [[0.5], [0.2], [0.1]],
    [[0.9], [0.1], [0.2]],
    [[0.3], [0.3], [0.3]],
    [[0.6], [0.2], [0.2]],
    [[0.4], [0.4], [0.1]],
    [[0.7], [0.2], [0.1]]
], dtype=float)

y = np.array([
    [0.6],
    [0.7],
    [0.8],
    [1.2],
    [0.9],
    [1.0],
    [0.9],
    [1.0]
], dtype=float)

In [ ]:
lstm = LSTMRegressor(
    input_size=1,
    hidden_size=12,
    output_size=1,
    learning_rate=0.01,
    epochs=500,
    random_state=42
)

lstm.fit(X, y)

predictions = lstm.predict(X)

print("Predictions:")
print(predictions)

print("MSE:", lstm.mse(X, y))
print("RMSE:", lstm.rmse(X, y))

In [ ]:
print(lstm.loss_history[:10])
print(lstm.loss_history[-10:])

In [ ]:
X_new = np.array([
    [[0.2], [0.2], [0.2]],
    [[0.5], [0.3], [0.2]],
    [[0.9], [0.4], [0.1]]
], dtype=float)

print(lstm.predict(X_new))